# N3 — RBM prior, CD-k and PCD

In [ ]:
from pathlib import Path
import sys, yaml, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
CFG=yaml.safe_load((ROOT/'config.yaml').read_text())
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:',DEVICE,'debug:',CFG['data']['debug'])

from torch.utils.data import DataLoader,TensorDataset
from piano_transcriber.priors import BernoulliRBM,train_rbm_epoch,train_rbm_epoch_pcd
from piano_transcriber.checkpoints import save_checkpoint\n# PCD alternative: persistent particles are carried from epoch to epoch.\npcd=BernoulliRBM(88,CFG['prior']['hidden_units']).to(DEVICE)\npersistent=None\nfor ep in range(min(10,CFG['prior']['epochs'])):\n loss,persistent=train_rbm_epoch_pcd(pcd,loader,DEVICE,k=CFG['prior']['cd_k'],lr=CFG['prior']['learning_rate'],persistent=persistent)\n if (ep+1)%5==0:print('PCD',ep+1,loss)\n

Use binary MIDI frame vectors from the training split only. The synthetic sample below validates code plumbing and must not be used as a reported result.

In [ ]:
train_frames=torch.bernoulli(torch.full((2048,88),.08))
loader=DataLoader(TensorDataset(train_frames),batch_size=CFG['prior']['batch_size'],shuffle=True)
rbm=BernoulliRBM(88,CFG['prior']['hidden_units']).to(DEVICE)
for ep in range(CFG['prior']['epochs']):
 loss=train_rbm_epoch(rbm,loader,DEVICE,CFG['prior']['cd_k'],CFG['prior']['learning_rate'])
 if (ep+1)%5==0:print(ep+1,loss)
save_checkpoint(ROOT/'models/rbm.pt',rbm,{'n_visible':88,'n_hidden':CFG['prior']['hidden_units']})\n# PCD alternative: persistent particles are carried from epoch to epoch.\npcd=BernoulliRBM(88,CFG['prior']['hidden_units']).to(DEVICE)\npersistent=None\nfor ep in range(min(10,CFG['prior']['epochs'])):\n loss,persistent=train_rbm_epoch_pcd(pcd,loader,DEVICE,k=CFG['prior']['cd_k'],lr=CFG['prior']['learning_rate'],persistent=persistent)\n if (ep+1)%5==0:print('PCD',ep+1,loss)\n

In [ ]:
with torch.no_grad():
 v=train_frames[:256].to(DEVICE);energy=rbm.free_energy(v).cpu().numpy()
 weights=rbm.W.cpu().numpy()
plt.hist(energy,bins=30);plt.title('Free energy');plt.show()
plt.figure(figsize=(12,3));plt.imshow(weights.T,aspect='auto',origin='lower');plt.title('Hidden-unit weights');plt.show()

For prior-only evaluation report held-out free-energy distributions. RBM free energy is not exact normalized NLL unless the partition function is estimated. For PCD, persist fantasy particles between updates rather than restarting chains.